In [1]:
import os
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as shc

In [5]:
if os.path.exists('CC_GENERAL.csv'):
    X = pd.read_csv('CC_GENERAL.csv').drop('CUST_ID', axis=1)
    X = X.ffill().bfill()                         # handle missing values
else:
    from sklearn.datasets import make_blobs
    data, _ = make_blobs(n_samples=600, n_features=8, centers=3, cluster_std=2.0, random_state=7)
    X = pd.DataFrame(data, columns=[f'f{i}' for i in range(8)])
print("Data shape:", X.shape)

X_scaled = StandardScaler().fit_transform(X)
X_norm = pd.DataFrame(normalize(X_scaled))
X_principal = pd.DataFrame(PCA(n_components=2).fit_transform(X_norm), columns=['P1', 'P2'])

Data shape: (8950, 17)


In [7]:
plt.figure(figsize=(8, 8)); plt.title('Visualising the data')
shc.dendrogram(shc.linkage(X_principal, method='ward'))
plt.savefig('a5_dendrogram.png'); plt.close()

In [8]:
ks, scores = [2, 3, 4, 5, 6], []
for k in ks:
    labels = AgglomerativeClustering(n_clusters=k).fit_predict(X_principal)
    scores.append(silhouette_score(X_principal, labels))
    plt.figure(figsize=(6, 6))
    plt.scatter(X_principal['P1'], X_principal['P2'], c=labels, cmap='rainbow')
    plt.title(f'k = {k}'); plt.savefig(f'a5_clusters_k{k}.png'); plt.close()
    print(f"k={k}  silhouette score = {scores[-1]:.4f}")

k=2  silhouette score = 0.4324
k=3  silhouette score = 0.4278
k=4  silhouette score = 0.3747
k=5  silhouette score = 0.3551
k=6  silhouette score = 0.3169


In [9]:
plt.bar(ks, scores); plt.xlabel('Number of clusters'); plt.ylabel('S(i)')
plt.savefig('a5_silhouette.png'); plt.close()
print(f"\nOptimal number of clusters (highest silhouette): {ks[int(np.argmax(scores))]}")


Optimal number of clusters (highest silhouette): 2
